# SDTM_Domain_Metadata_Provenance — the Provenance sheet of a hand-kept file

`sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx` is edited by hand (its sheets are
reference data and, in `Variable_Meaning`, our position). This notebook writes its `Provenance` sheet,
as the generated reference files carry one: the pinned source versions, when the sheet was written, and
per sheet its rows, columns and a content hash. A consumer can then recompute the hash and see whether
the data changed.

**Run it after every hand edit of the file.** `domain-behaviour/notebooks/Domain_Behaviour.ipynb` stops
when a hash does not match its sheet, so a stale Provenance cannot pass unnoticed.

It changes no other sheet. The hash method is the one of the other reference files: SHA-256 (UTF-8) of
the sheet read as text, empty cells as "" (pandas `read_excel`, `dtype=str`): header and rows, cells
joined by tab, rows by newline. The `ReadMe` sheet is not hashed.

The versions are the pins in force when the sheet is written (`Reference_Versions.json`); the file is
checked against SDTMIG v3.4 and SDTM CT by `sdtm-findings-graph/notebooks/Scope_Check.ipynb`, not here.

## 1. Setup

In [ ]:
import hashlib
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

REPO_ROOT = Path.cwd().parent.parent   # notebooks/ -> sdtm-domain-reference/ -> repo root
VERSIONS = json.loads((REPO_ROOT / 'Reference_Versions.json').read_text(encoding='utf-8'))
META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
NOTEBOOK = 'sdtm-domain-reference/notebooks/SDTM_Domain_Metadata_Provenance.ipynb'
VERSION_KEYS = ['SDTM_CT', 'COSMoS', 'SDTMIG', 'SDTM', 'NCIt']
METHOD = ('SHA-256 (UTF-8) of the sheet read as text, empty cells as "" (pandas read_excel, dtype=str): '
          'header and rows, cells joined by tab, rows by newline. Equal hash = equal data.')
if not META_FILE.exists():
    raise FileNotFoundError(META_FILE)
print(META_FILE.relative_to(REPO_ROOT))

## 2. Hash every data sheet and write Provenance

An existing `Provenance` sheet is replaced; the other sheets are read, not written.

In [ ]:
def sheet_sha256(df):
    rows = ['\t'.join(df.columns)] + ['\t'.join(r) for r in df.itertuples(index=False, name=None)]
    return hashlib.sha256('\n'.join(rows).encode('utf-8')).hexdigest()


sheets = pd.read_excel(META_FILE, sheet_name=None, dtype=str)
rows = [(k, VERSIONS[k]) for k in VERSION_KEYS]
rows += [('Generated', f'{datetime.now():%Y-%m-%d %H:%M}'), ('Notebook', NOTEBOOK)]
for name, df in sheets.items():
    if name in ('ReadMe', 'Provenance'):
        continue
    df = df.fillna('')
    rows += [(f'{name}.Rows', str(len(df))), (f'{name}.Columns', '|'.join(df.columns)),
             (f'{name}.Content_SHA256', sheet_sha256(df))]
rows.append(('Content_SHA256.Method', METHOD))

wb = load_workbook(META_FILE)
if 'Provenance' in wb.sheetnames:
    del wb['Provenance']
ws = wb.create_sheet('Provenance')
for ci, head in enumerate(['Key', 'Value'], 1):
    cell = ws.cell(row=1, column=ci, value=head)
    cell.font = Font(name='Arial', bold=True, size=10, color='FFFFFF')
    cell.fill = PatternFill('solid', fgColor='808080')
for ri, (k, v) in enumerate(rows, 2):
    ws.cell(row=ri, column=1, value=k).font = Font(name='Arial', size=10)
    ws.cell(row=ri, column=2, value=v).font = Font(name='Arial', size=10)
ws.column_dimensions['A'].width = 34
ws.column_dimensions['B'].width = 90
ws.freeze_panes = 'A2'
wb.save(META_FILE)
print(f'Provenance: {len(rows)} keys')
for k, v in rows:
    if k.endswith('.Rows'):
        print(f'  {k[:-5]}: {v} rows')

## 3. Check

Read the file back and recompute every hash; the run fails if one differs.

In [ ]:
prov = dict(pd.read_excel(META_FILE, sheet_name='Provenance', dtype=str).values)
back = pd.read_excel(META_FILE, sheet_name=None, dtype=str)
bad = [n for n, df in back.items() if n not in ('ReadMe', 'Provenance')
       and prov.get(f'{n}.Content_SHA256') != sheet_sha256(df.fillna(''))]
if bad:
    raise RuntimeError(f'Hashes do not match after writing: {bad}')
print('Provenance matches every data sheet.')